In [2]:
import numpy as np
import time

def get_paper_Qk_sequence(k, horizon):
    """
    【核心修复】严格按照原论文定义的 Sieve Operator: S_p = R L^{p-1}
    素数 p 的所有倍数（包含 p 本身！）都被标记为 R (1)
    """
    primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 
              43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101]
    used_primes = primes[:k]
    
    seq = np.zeros(horizon, dtype=np.int8) # 0 代表 L (存活)
    seq[0] = 1 # 0 永远是 R
    
    for p in used_primes:
        seq[0:horizon:p] = 1 # 周期算子：倍数全杀，包含质数 p 自身！
        
    return seq, primes[k]

def calculate_defect_rate(seq):
    N = len(seq)
    defect_count = 0
    cumsum_R = np.cumsum(seq)
    
    for shift in range(1, N):
        orig_sub = seq[:N-shift]
        shift_sub = seq[shift:]
        
        diffs = (orig_sub != shift_sub)
        if not np.any(diffs):
            continue
            
        first_diff = np.argmax(diffs)
        r_count = cumsum_R[first_diff - 1] if first_diff > 0 else 0
        
        orig_val = orig_sub[first_diff]
        shift_val = shift_sub[first_diff]
        
        base_cmp = -1 if orig_val < shift_val else 1
        real_cmp = -base_cmp if (r_count % 2 == 1) else base_cmp
            
        if real_cmp == -1: # 发生拓扑逃逸
            defect_count += 1
            
    return defect_count, defect_count / (N - 1) if N > 1 else 0

def run_experiment():
    print("🚀 启动实验：论文 Q_k 筛法轨道的渐近合法性 (Asymptotic Admissibility)\n")
    primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]
              
    print(f"{'级数 k':<6} | {'引入素数':<8} | {'物理视界 N':<10} | {'拓扑缺陷数':<12} | {'缺陷密度 ρ(N)':<15}")
    print("-" * 65)
    
    for k in range(1, 15):
        next_p = primes[k]
        horizon = next_p ** 2  # 论文声称的合法物理视界
        
        seq, _ = get_paper_Qk_sequence(k, horizon)
        defects, density = calculate_defect_rate(seq)
        
        print(f"Q_{k:<4} | {primes[k-1]:<8} | {horizon:<10} | {defects:<12} | {density:.6f}")

if __name__ == "__main__":
    run_experiment()

🚀 启动实验：论文 Q_k 筛法轨道的渐近合法性 (Asymptotic Admissibility)

级数 k   | 引入素数     | 物理视界 N     | 拓扑缺陷数        | 缺陷密度 ρ(N)      
-----------------------------------------------------------------
Q_1    | 2        | 9          | 0            | 0.000000
Q_2    | 3        | 25         | 0            | 0.000000
Q_3    | 5        | 49         | 1            | 0.020833
Q_4    | 7        | 121        | 0            | 0.000000
Q_5    | 11       | 169        | 1            | 0.005952
Q_6    | 13       | 289        | 0            | 0.000000
Q_7    | 17       | 361        | 0            | 0.000000
Q_8    | 19       | 529        | 0            | 0.000000
Q_9    | 23       | 841        | 0            | 0.000000
Q_10   | 29       | 961        | 0            | 0.000000
Q_11   | 31       | 1369       | 0            | 0.000000
Q_12   | 37       | 1681       | 0            | 0.000000
Q_13   | 41       | 1849       | 0            | 0.000000
Q_14   | 43       | 2209       | 0            | 0.000000
